In [ ]:
import matplotlib.pyplot as plt
import mne
import numpy as np
import torch
from src.features import extract_band_powers
from src.model import EEGNetLight
from src.preprocess import EEGPreprocessor

# Set seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)


# 1. Load Sample EEG Data
# Loading the standard MNE sample dataset (auditory/visual experiment).

data_path = mne.datasets.sample.data_path()
raw_fname = data_path / "MEG" / "sample" / "sample_audvis_raw.fif"

# Load raw signal and pick only 32 EEG channels for benchmarking
raw = mne.io.read_raw_fif(raw_fname, preload=True, verbose=False)
raw.pick_types(meg=False, eeg=True, eog=True)
raw.crop(tmin=0, tmax=60)  # Use first 60 seconds for fast demo

print(
    f"Loaded {len(raw.ch_names)} channels | Sampling rate: {raw.info['sfreq']} Hz"
)


# 2. Bandpass and Notch Filtering

preprocessor = EEGPreprocessor(l_freq=0.5, h_freq=45.0, notch_freq=50.0)
raw_filtered = preprocessor.apply_filters(raw)


#  3. ICA Decomposition & Ocular Artifact Removal

# Fit FastICA on the high-pass filtered signal
preprocessor.fit_ica(raw_filtered, n_components=15)

# Detect and exclude EOG-related components
raw_cleaned = preprocessor.remove_eog_artifacts(
    raw_filtered, eog_channel="EOG 061"
)

# Visualize ICA spatial components
preprocessor.ica.plot_components(picks=range(min(6, 15)), show=True)

# 4. Quantitative Feature Extraction (Band Powers)
band_powers = extract_band_powers(raw_cleaned)
for band, power in band_powers.items():
    print(
        f"{band} Band ({band}) Power - Mean: {power.mean():.4e}, Std: {power.std():.4e}"
    )


# Define dummy batch for model verification: (Batch, 1, Channels, Samples)
# Epoch window: 2 seconds @ 128 Hz resampling
n_channels = 32
n_samples = 256
batch_size = 16

model = EEGNetLight(
    num_channels=n_channels, num_classes=2, samples=n_samples
)
dummy_eeg = torch.randn(batch_size, 1, n_channels, n_samples)

with torch.no_grad():
    predictions = model(dummy_eeg)

print(f"Input batch shape:  {dummy_eeg.shape}")
print(f"Output logit shape: {predictions.shape}")
print("Pipeline verified successfully.")
